# Importar y extraer las variables locales de ERA5

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

In [2]:
PROJECT_ROOT = Path.cwd().parent

ERA5_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "era5_land_monthly.nc"
)

ds = xr.open_dataset(ERA5_FILE)

ds

<xarray.Dataset> Size: 90MB
Dimensions:     (valid_time: 912, latitude: 66, longitude: 31)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 7kB 1950-01-01 ... 2025-12-01
    expver      (valid_time) <U4 15kB ...
  * latitude    (latitude) float64 528B 1.5 1.4 1.3 1.2 ... -4.7 -4.8 -4.9 -5.0
  * longitude   (longitude) float64 248B -81.5 -81.4 -81.3 ... -78.7 -78.6 -78.5
    number      int64 8B ...
Data variables:
    d2m         (valid_time, latitude, longitude) float32 7MB ...
    t2m         (valid_time, latitude, longitude) float32 7MB ...
    swvl1       (valid_time, latitude, longitude) float32 7MB ...
    swvl2       (valid_time, latitude, longitude) float32 7MB ...
    swvl3       (valid_time, latitude, longitude) float32 7MB ...
    swvl4       (valid_time, latitude, longitude) float32 7MB ...
    ro          (valid_time, latitude, longitude) float32 7MB ...
    e           (valid_time, latitude, longitude) float32 7MB ...
    u10         (valid_time, latitude, longitude) float32 7MB ...
    v10         (valid_time, latitude, longitude) float32 7MB ...
    sp          (valid_time, latitude, longitude) float32 7MB ...
    tp          (valid_time, latitude, longitude) float32 7MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-09-16T18:21 GRIB to CDM+CF via cfgrib-0.9.1...

usar la misma region costera para target

In [3]:
coast = ds.sel(
    latitude=slice(1.5, -5.0),
    longitude=slice(-81.5, -79.0)
)

creacion de la mascara terrestre

In [4]:
land_mask = (
    coast["tp"]
    .isel(valid_time=0)
    .notnull()
)

In [5]:
print(
    "Celdas terrestres:",
    int(land_mask.sum())
)

Celdas terrestres: 965


rEVISION DE UNIDADES ANTES DE TRANSFORMACION

In [6]:
variables = [
    "t2m",
    "d2m",
    "u10",
    "v10",
    "sp",
    "e",
    "ro",
    "swvl1",
    "swvl2",
    "swvl3",
    "swvl4",
]

for var in variables:
    print(
        var,
        "->",
        coast[var].attrs.get("long_name"),
        "|",
        coast[var].attrs.get("units")
    )

t2m -> 2 metre temperature | K
d2m -> 2 metre dewpoint temperature | K
u10 -> 10 metre U wind component | m s**-1
v10 -> 10 metre V wind component | m s**-1
sp -> Surface pressure | Pa
e -> Evaporation | m of water equivalent
ro -> Runoff | m
swvl1 -> Volumetric soil water layer 1 | m**3 m**-3
swvl2 -> Volumetric soil water layer 2 | m**3 m**-3
swvl3 -> Volumetric soil water layer 3 | m**3 m**-3
swvl4 -> Volumetric soil water layer 4 | m**3 m**-3


funcion para obtener el promedio costero

In [7]:
def coastal_mean(dataset, variable, mask):
    """
    Calcula la media espacial mensual de una variable
    sobre las celdas terrestres de la región costera.
    """
    return (
        dataset[variable]
        .where(mask)
        .mean(
            dim=["latitude", "longitude"],
            skipna=True
        )
    )

In [8]:
t2m_mean = coastal_mean(
    coast,
    "t2m",
    land_mask
)

trasnformaciones de variables de temperatura de de K a C

In [9]:
t2m_c = (
    coastal_mean(coast, "t2m", land_mask)
    - 273.15
)

d2m_c = (
    coastal_mean(coast, "d2m", land_mask)
    - 273.15
)

In [10]:
t2m_c.name = "temperature_c"
d2m_c.name = "dewpoint_c"

In [11]:
print(float(t2m_c.mean()))
print(float(d2m_c.mean()))

21.060832977294922
17.60191535949707


para el viento

In [12]:
u10_mean = coastal_mean(
    coast, "u10", land_mask
)

v10_mean = coastal_mean(
    coast, "v10", land_mask
)

u10_mean.name = "u10_ms"
v10_mean.name = "v10_ms"

crear variable velocidad viento

In [13]:
wind_speed = (
    np.hypot(
        coast["u10"],
        coast["v10"]
    )
    .where(land_mask)
    .mean(
        dim=["latitude", "longitude"],
        skipna=True
    )
)

wind_speed.name = "wind_speed_ms"

presion suoerficial

In [14]:
surface_pressure = (
    coastal_mean(
        coast,
        "sp",
        land_mask
    )
    / 100
)

surface_pressure.name = "surface_pressure_hpa"

runoff varaible hidrologica acumulada

In [15]:
days_in_month = (
    coast["valid_time"]
    .dt.days_in_month
)

runoff_mm = (
    coastal_mean(
        coast,
        "ro",
        land_mask
    )
    * 1000
    * days_in_month
)

runoff_mm.name = "runoff_mm"

evaporacion

In [16]:
evaporation_mm = (
    -coastal_mean(
        coast,
        "e",
        land_mask
    )
    * 1000
    * days_in_month
)

evaporation_mm.name = "evaporation_mm"

In [17]:
print(
    "Evaporation raw mean:",
    float(
        coastal_mean(
            coast,
            "e",
            land_mask
        ).mean()
    )
)

Evaporation raw mean: -0.0024010580964386463


humedad del suelo (no se necesita conversion)

In [18]:
soil_water_1 = coastal_mean(
    coast, "swvl1", land_mask
)

soil_water_2 = coastal_mean(
    coast, "swvl2", land_mask
)

soil_water_3 = coastal_mean(
    coast, "swvl3", land_mask
)

soil_water_4 = coastal_mean(
    coast, "swvl4", land_mask
)

In [19]:
soil_water_1.name = "soil_water_1"
soil_water_2.name = "soil_water_2"
soil_water_3.name = "soil_water_3"
soil_water_4.name = "soil_water_4"

Construccion del dataset de variables locales

In [20]:
local_ds = xr.Dataset({
    "temperature_c": t2m_c,
    "dewpoint_c": d2m_c,
    "u10_ms": u10_mean,
    "v10_ms": v10_mean,
    "wind_speed_ms": wind_speed,
    "surface_pressure_hpa": surface_pressure,
    "evaporation_mm": evaporation_mm,
    "runoff_mm": runoff_mm,
    "soil_water_1": soil_water_1,
    "soil_water_2": soil_water_2,
    "soil_water_3": soil_water_3,
    "soil_water_4": soil_water_4,
})

In [21]:
local_ds

<xarray.Dataset> Size: 73kB
Dimensions:               (valid_time: 912)
Coordinates:
  * valid_time            (valid_time) datetime64[ns] 7kB 1950-01-01 ... 2025...
    expver                (valid_time) <U4 15kB '0001' '0001' ... '0001' '0001'
    number                int64 8B 0
Data variables:
    temperature_c         (valid_time) float32 4kB 20.85 21.18 ... 21.31 21.82
    dewpoint_c            (valid_time) float32 4kB 17.14 17.73 ... 17.89 16.74
    u10_ms                (valid_time) float32 4kB 0.612 0.4586 ... 0.8369
    v10_ms                (valid_time) float32 4kB 0.438 0.2052 ... 0.4112
    wind_speed_ms         (valid_time) float32 4kB 0.9346 0.6798 ... 1.265 1.197
    surface_pressure_hpa  (valid_time) float32 4kB 932.0 932.3 ... 932.5 932.2
    evaporation_mm        (valid_time) float64 7kB 73.82 68.22 ... 71.78 73.63
    runoff_mm             (valid_time) float64 7kB 77.13 127.1 ... 85.25 44.82
    soil_water_1          (valid_time) float32 4kB 0.367 0.3797 ... 0.3045
    soil_water_2          (valid_time) float32 4kB 0.3506 0.3762 ... 0.3194
    soil_water_3          (valid_time) float32 4kB 0.3276 0.3557 ... 0.3302
    soil_water_4          (valid_time) float32 4kB 0.3774 0.3823 ... 0.3954

In [22]:
local_df = (
    local_ds
    .to_dataframe()
    .reset_index()
    .rename(
        columns={
            "valid_time": "date"
        }
    )
)

In [23]:
local_df.head()

,date,number,expver,temperature_c,dewpoint_c,u10_ms,v10_ms,wind_speed_ms,surface_pressure_hpa,evaporation_mm,runoff_mm,soil_water_1,soil_water_2,soil_water_3,soil_water_4
0,1950-01-01,0,0001,20.845703,17.144318,0.612021,0.437981,0.934633,932.041931,73.816581,77.133245,0.366950,0.350620,0.327577,0.377396
1,1950-02-01,0,0001,21.176575,17.730072,0.458641,0.205236,0.679810,932.303284,68.222015,127.091787,0.379683,0.376162,0.355740,0.382328
2,1950-03-01,0,0001,21.265747,17.873138,0.550594,0.218168,0.734493,932.719360,90.915420,120.328326,0.387393,0.393051,0.388167,0.396222
3,1950-04-01,0,0001,20.958221,18.018341,0.366579,0.398198,0.798530,932.152405,83.923831,185.992613,0.405306,0.401859,0.394797,0.402872
4,1950-05-01,0,0001,20.652649,16.798309,0.487092,0.486566,0.953303,932.679749,80.675658,92.131619,0.349731,0.361851,0.368473,0.402260


In [24]:
print(local_df.shape)

print(
    local_df["date"].min(),
    local_df["date"].max()
)

print(
    local_df.isna().sum()
)

(912, 15)
1950-01-01 00:00:00 2025-12-01 00:00:00
date                    0
number                  0
expver                  0
temperature_c           0
dewpoint_c              0
u10_ms                  0
v10_ms                  0
wind_speed_ms           0
surface_pressure_hpa    0
evaporation_mm          0
runoff_mm               0
soil_water_1            0
soil_water_2            0
soil_water_3            0
soil_water_4            0
dtype: int64


In [25]:
local_df.describe().T

,count,mean,min,25%,50%,75%,max,std
date,912,1987-12-16 11:00:00,1950-01-01 00:00:00,1968-12-24 06:00:00,1987-12-16 12:00:00,2006-12-08 18:00:00,2025-12-01 00:00:00,NaN
number,912.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
temperature_c,912.0,21.060833,19.248138,20.544502,21.053909,21.556183,23.241638,0.707282
dewpoint_c,912.0,17.601915,14.372742,16.544205,17.519745,18.64753,21.01178,1.269574
u10_ms,912.0,0.427332,-0.091076,0.287948,0.388044,0.562075,1.012087,0.204611
v10_ms,912.0,0.448035,-0.068113,0.310904,0.480639,0.589294,0.810154,0.175484
wind_speed_ms,912.0,1.020065,0.275858,0.77472,1.08939,1.258433,1.473814,0.275035
surface_pressure_hpa,912.0,932.206238,929.675049,931.769882,932.238953,932.665833,933.794678,0.646194
evaporation_mm,912.0,73.016524,19.3765,65.077707,71.77911,81.930013,101.467738,12.658957
runoff_mm,912.0,153.856883,13.492688,69.781377,113.691792,213.199117,748.4214,116.168975


In [26]:
local_df = local_df.drop(
    columns=["number", "expver"],
    errors="ignore"
)

local_df.shape

(912, 13)

GUARDAR LOS DATOS

In [27]:
OUTPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "era5_local_features.csv"
)

local_df.to_csv(
    OUTPUT_FILE,
    index=False
)